# SSAFY AI 챌린지 — Colab 학습 연장 (세션 재개 가능)

노트북 RTX 5060(8GB)에서는 7B 학습이 불가능해 Colab에서 이어서 학습합니다.

**현재 상태**: 어댑터 `lora_7b_ft_check` 는 104 optimizer step(1 epoch의 19%)에서
멈춰 있고, 그 상태로 홀드아웃 +2.75%p / LB 0.84430 → 0.86322 를 냈습니다.

**이 노트북이 하는 일**
1. Drive에서 train 이미지와 어댑터를 가져온다
2. 옵티마이저·스케줄러·step 까지 저장/복원하며 학습한다 — **세션이 끊겨도 정확히 이어집니다**
3. 홀드아웃 609건으로 재평가해 좋아졌을 때만 채택한다

**세션이 끊기면** 런타임을 새로 잡고 셀을 위에서부터 다시 실행하세요.
`train_state.pt` 에서 자동으로 이어집니다.

**끝나면** Drive의 `lora_colab_latest` 를 노트북(로컬)으로 내려받아
`python infer_ft.py --test` 를 돌리면 됩니다. test 이미지는 Colab에 올릴 필요가 없습니다.

## 0. 런타임 확인 — GPU가 없으면 여기서 멈추세요

In [ ]:
import torch, subprocess
assert torch.cuda.is_available(), "GPU 런타임이 아닙니다. 런타임 > 런타임 유형 변경 > GPU"
name = torch.cuda.get_device_name()
cap  = torch.cuda.get_device_capability()
free, total = torch.cuda.mem_get_info()
BF16 = cap[0] >= 8          # ★ is_bf16_supported() 는 에뮬레이션도 True라 쓰면 안 됨
print(f"GPU  : {name}  cc={cap}")
print(f"VRAM : {total/1e9:.1f} GB (여유 {free/1e9:.1f} GB)")
print(f"bf16 : {BF16} (하드웨어 지원. cc 8.0 이상에서만 True)")
if total/1e9 < 14:
    print("\n⚠️ VRAM 14GB 미만입니다. 7B 4bit + 640px 학습은 어렵습니다.")
if not BF16:
    print("\n⚠️ bf16 미지원(T4) — fp16으로 돕니다. 데스크탑은 bf16이었으므로 조건이 다릅니다.")
    print("   학습 셀이 GradScaler를 켜고, 5 step 뒤 가중치가 실제로 움직였는지 검사합니다.")
    print("   (fp16은 그래디언트가 0으로 언더플로해 '조용히 학습이 안 되는' 경우가 있습니다)")

## 1. 설치 (2~3분)

In [ ]:
!pip -q install "transformers>=4.49.0,<5.0.0" "accelerate>=0.34" "peft>=0.13" \
                "bitsandbytes>=0.43" "safetensors" 2>&1 | tail -2
import transformers, peft, bitsandbytes
print("transformers", transformers.__version__, "/ peft", peft.__version__,
      "/ bnb", bitsandbytes.__version__)

## 2. 데이터 — Drive에서 가져오기

Drive의 `MyDrive/ssafy-vqa/` 에 아래가 있어야 합니다.

```
train_images.zip      train 이미지 5,073장 (621MB)
train.csv
holdout_ids.csv       홀드아웃 609건 id — 순서가 중요합니다
lora_7b_ft_check/     현재 어댑터 (adapter_model.safetensors + adapter_config.json)
```

`test.csv` 와 test 이미지는 **필요 없습니다** — 추론은 로컬 노트북에서 합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil, time

DRIVE = Path("/content/drive/MyDrive/ssafy-vqa")
ROOT  = Path("/content/work"); ROOT.mkdir(exist_ok=True)
assert DRIVE.exists(), f"{DRIVE} 가 없습니다. Drive 경로를 확인하세요."

for n in ["train.csv", "holdout_ids.csv"]:
    assert (DRIVE / n).exists(), f"{n} 가 Drive에 없습니다"
    shutil.copy(DRIVE / n, ROOT / n)

# 이미지 압축 해제 (세션마다 한 번, 2~4분)
if not (ROOT / "train").exists():
    t = time.time()
    !unzip -q "{DRIVE}/train_images.zip" -d "{ROOT}"
    print(f"압축 해제 {time.time()-t:.0f}초")
n_img = len(list((ROOT / "train").glob("*.jpg")))
print(f"train 이미지 {n_img}장")
assert n_img > 5000, f"이미지가 {n_img}장뿐입니다 — zip을 확인하세요"

## 3. 설정

In [ ]:
MODEL_ID   = "Qwen/Qwen2.5-VL-7B-Instruct"
# ★ 리비전 고정 — 데스크탑이 받은 것과 바이트 단위로 같은 가중치를 받습니다.
#   QLoRA는 베이스 모델을 얼려두고 어댑터만 학습하므로, 베이스가 조금이라도 다르면
#   어댑터가 '다른 가중치 위에' 얹혀 학습 결과가 어긋납니다.
#   HF 저장소의 main 브랜치는 언제든 갱신될 수 있으므로 해시로 못박습니다.
REVISION   = "cc594898137f460bfe9f0759e9844b3ce807cfb5"
IMAGE_PX   = 640         # ★ 어댑터가 640px로 학습됨. 바꾸면 이전 학습이 무의미해집니다
SEED, VAL_RATIO = 42, 0.12

# ---------- 학습 ----------
FT_BATCH   = 1           # 7B 4bit + 640px 는 16GB에서도 1이 한계
FT_ACCUM   = 8           # 유효 배치 8 (데스크탑과 동일)
FT_LR      = 1e-4
FT_MAX_MIN = 75          # 이 세션에서 학습할 시간(분). Colab이 끊기기 전에 저장되도록 넉넉히
SAVE_EVERY = 20          # optimizer step 마다 Drive에 상태 저장 (끊김 대비)

# 총 학습 목표. 1 epoch = ceil(4464/8) = 558 step, 이미 104 step 완료했습니다.
#
# ★ 558(1 epoch 완주)이 아니라 300으로 잡은 이유:
#   T4는 데스크탑(5060 Ti)보다 3~4배 느려 558까지는 9시간 = 무료 세션 7~8회입니다.
#   더 중요한 건 코사인 스케줄이라 **목표 step에서 학습률이 0이 됩니다.** 558로 잡고
#   300에서 멈추면 학습률이 높은 상태로 끝나 어댑터가 어정쩡해집니다.
#   달성 가능한 목표를 잡아 제대로 감쇠시키는 편이 낫습니다.
#   300까지 가서도 계속 좋아지면 그때 TARGET_STEPS를 올리세요 (스케줄이 자동 재구성됩니다).
TARGET_STEPS = 300       # 104 -> 300 (1 epoch의 54%). T4에서 약 4시간 = 세션 3~4회
DONE_BEFORE  = 104       # 데스크탑에서 이미 한 step 수 (state 파일이 있으면 무시됨)

# ---------- 경로 ----------
SEED_ADAPTER = DRIVE / "lora_7b_ft_check"     # 최초 1회만 사용 (데스크탑 결과)
WORK         = DRIVE / "colab_train"          # 이 노트북의 산출물
WORK.mkdir(exist_ok=True)
ADAPTER_OUT  = WORK / "lora_colab_latest"
STATE_PATH   = WORK / "train_state.pt"

# ★ torch.cuda.is_bf16_supported() 를 쓰면 안 됩니다.
#   기본값이 including_emulation=True 라 T4(cc 7.5)에서도 True를 반환하는데,
#   T4에는 bf16 텐서코어가 없어 소프트웨어 에뮬레이션으로 돌아 **8배 느려집니다**
#   (실측 161 s/step = 20 s/샘플. 데스크탑 bf16 네이티브는 2.53 s/샘플).
#   하드웨어 bf16은 compute capability 8.0(Ampere) 이상에서만 지원됩니다.
_CC = torch.cuda.get_device_capability()
COMPUTE_DTYPE = torch.bfloat16 if _CC[0] >= 8 else torch.float16
print(f"compute capability {_CC} -> {COMPUTE_DTYPE}")
print("상태 파일     :", STATE_PATH, "(있으면 이어서, 없으면 새로)")
print("이번 세션 상한:", FT_MAX_MIN, "분")

## 4. 라이브러리 · 데이터

In [ ]:
import os, math, random, gc, json
import numpy as np, pandas as pd
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from typing import Any
from PIL import Image, ImageEnhance
from tqdm.auto import tqdm
from transformers import (AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig,
                          get_cosine_schedule_with_warmup)
from peft import (LoraConfig, get_peft_model, prepare_model_for_kbit_training,
                  set_peft_model_state_dict, get_peft_model_state_dict)
from safetensors.torch import load_file

Image.MAX_IMAGE_PIXELS = None
device = "cuda"
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

train_df = pd.read_csv(ROOT / "train.csv")
CNT_PAT = r"몇 개|몇개|개입니까|개인가요|몇 가지"
def is_counting(s): return s.str.contains(CNT_PAT, regex=True)

# 홀드아웃은 저장된 id 목록을 씁니다 (sklearn 버전 차이에 영향받지 않음)
hold_ids = pd.read_csv(ROOT / "holdout_ids.csv")["id"].tolist()
va_part = train_df.set_index("id").loc[hold_ids].reset_index()
tr_part = train_df[~train_df["id"].isin(hold_ids)].reset_index(drop=True)
tr_part["weight"] = 1.0
assert len(va_part) == 609 and len(tr_part) == len(train_df) - 609
print(f"학습 {len(tr_part)} / 홀드아웃 {len(va_part)}")
print(f"홀드아웃 카운팅 비율 {100*is_counting(va_part.question).mean():.1f}% (test 33.8%)")

## 5. 프롬프트 · Dataset · Collator — 데스크탑과 한 글자도 다르면 안 됩니다

In [ ]:
SYSTEM_INSTRUCT = ("You are a helpful visual question answering assistant. "
                   "Answer using exactly one letter among a, b, c, or d. No explanation.")

def build_mc_prompt(question, a, b, c, d):
    return (f"{question}\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
            "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.")

def build_messages(row, img):
    o = [str(row[k]) for k in "abcd"]
    return [{"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user", "content": [{"type": "image", "image": img},
             {"type": "text", "text": build_mc_prompt(str(row["question"]), *o)}]}]

def augment(img):
    if random.random() < 0.7: img = ImageEnhance.Brightness(img).enhance(random.uniform(0.75, 1.25))
    if random.random() < 0.7: img = ImageEnhance.Contrast(img).enhance(random.uniform(0.80, 1.20))
    if random.random() < 0.5: img = ImageEnhance.Color(img).enhance(random.uniform(0.80, 1.20))
    if random.random() < 0.3:
        img = img.rotate(random.uniform(-8, 8), resample=Image.BILINEAR, fillcolor=(255, 255, 255))
    return img

class VQADataset(Dataset):
    def __init__(self, df, train=True, aug=False):
        self.df = df.reset_index(drop=True); self.train = train; self.aug = aug
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(ROOT / row["path"]).convert("RGB")
        if self.aug: img = augment(img)
        item = {"messages": build_messages(row, img), "image": img,
                "weight": float(row["weight"]) if "weight" in row else 1.0}
        if self.train: item["gold"] = str(row["answer"]).strip().lower()
        return item

@dataclass
class Collator:
    processor: Any
    train: bool = True
    def __call__(self, batch):
        texts, images, weights = [], [], []
        for s in batch:
            t = self.processor.apply_chat_template(s["messages"], tokenize=False,
                                                   add_generation_prompt=True)
            if self.train: t = t + s["gold"]      # <|im_end|> 를 붙이지 않는다
            texts.append(t); images.append(s["image"]); weights.append(s["weight"])
        # 학습은 right padding(마지막 실제 토큰 = 정답), 추론은 left padding
        self.processor.tokenizer.padding_side = "right" if self.train else "left"
        enc = self.processor(text=texts, images=images, padding=True, return_tensors="pt")
        if self.train:
            ids, am = enc["input_ids"], enc["attention_mask"]
            last = am.sum(1) - 1
            labels = torch.full_like(ids, -100)
            r = torch.arange(ids.size(0))
            labels[r, last] = ids[r, last]
            enc["labels"] = labels; enc["_last"] = last
        enc["_weight"] = torch.tensor(weights, dtype=torch.float32)
        return enc
print("정의 완료")

## 6. 모델 로드 (5~10분) — 어댑터는 최신 상태에서 이어받습니다

In [ ]:
# ★ 이전 모델을 확실히 풀어야 합니다. `del model` 만으로는 부족합니다 —
#   학습 셀이 만든 opt(옵티마이저)가 파라미터를 붙잡고 있어 메모리가 안 풀리고,
#   다음 로드에서 OOM 납니다 (prepare_model_for_kbit_training 이 embed_tokens 를
#   fp32로 승격하며 2GB를 더 요구하는 지점에서 터집니다).
for _v in ["model", "base_model", "opt", "sch", "params", "loader", "pbar",
           "scaler", "p_new", "batch", "loss"]:
    if _v in globals(): del globals()[_v]
# IPython 출력 히스토리(_, __, Out)도 텐서를 붙잡습니다
try:
    from IPython import get_ipython
    _ip = get_ipython()
    if _ip is not None:
        _ip.user_ns.pop("_", None); _ip.user_ns.pop("__", None)
        if hasattr(_ip, "displayhook"): _ip.displayhook.flush()
except Exception: pass
gc.collect(); torch.cuda.empty_cache()
_free0 = torch.cuda.mem_get_info()[0] / 1e9
print(f"로드 전 GPU 여유 {_free0:.1f} GB")
if _free0 < 10:
    raise RuntimeError(
        f"여유가 {_free0:.1f} GB뿐입니다. 이전 모델이 안 풀렸습니다. "
        "런타임 > 세션 다시 시작(Restart session) 후 셀 0부터 다시 실행하세요. "
        "'연결 해제 및 삭제'가 아닙니다 — 세션 재시작은 디스크를 유지하므로 재다운로드가 없습니다.")

processor = AutoProcessor.from_pretrained(MODEL_ID, revision=REVISION,
                                          min_pixels=IMAGE_PX*IMAGE_PX,
                                          max_pixels=IMAGE_PX*IMAGE_PX, trust_remote_code=True)
tok = processor.tokenizer

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID, revision=REVISION,
    device_map={"": 0}, trust_remote_code=True, torch_dtype=COMPUTE_DTYPE,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=COMPUTE_DTYPE))

bad = [n for n, p in base_model.named_parameters() if p.device.type in ("meta", "cpu")]
assert not bad, f"GPU 밖 파라미터 {len(bad)}개 — VRAM 부족으로 오프로드됨"

base_model = prepare_model_for_kbit_training(
    base_model, use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False})
base_model.config.use_cache = False

model = get_peft_model(base_model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM"))
# 학습 파라미터는 반드시 fp32 (fp16/bf16이면 lr 1e-4 업데이트가 사라집니다)
for _, p in model.named_parameters():
    if p.requires_grad and p.dtype != torch.float32: p.data = p.data.float()
model.print_trainable_parameters()

# ---- 어댑터 이어받기: colab_latest 가 있으면 그것을, 없으면 데스크탑 것을 ----
src = ADAPTER_OUT if (ADAPTER_OUT / "adapter_model.safetensors").exists() else SEED_ADAPTER
assert (src / "adapter_model.safetensors").exists(), f"{src} 에 어댑터가 없습니다"
set_peft_model_state_dict(model, load_file(str(src / "adapter_model.safetensors")))
nz = sum(float(p.detach().abs().sum()) for n, p in model.named_parameters() if "lora_B" in n)
assert nz > 0, "lora_B가 전부 0 — 어댑터가 반영되지 않았습니다"
print(f"✅ 어댑터 이어받기: {src.name}  (lora_B 절대합 {nz:.1f})")

# 실제로 어떤 리비전이 쓰였는지 확인 (고정이 먹었는지 눈으로 보기)
import glob, os
_snap = glob.glob(os.path.expanduser(
    "~/.cache/huggingface/hub/models--Qwen--Qwen2.5-VL-7B-Instruct/snapshots/*"))
print("사용된 스냅샷:", [os.path.basename(s) for s in _snap])
assert any(REVISION in s for s in _snap), "고정한 리비전이 아닙니다"

CHOICES = list("abcd")
CHOICE_IDS = []
for ch in CHOICES:
    ids = tok.encode(ch, add_special_tokens=False)
    assert len(ids) == 1, f"'{ch}' 단일 토큰 아님"
    CHOICE_IDS.append(ids[0])
CHOICE_IDS = torch.tensor(CHOICE_IDS)
CH = np.array(CHOICES)
print("선지 토큰", dict(zip(CHOICES, CHOICE_IDS.tolist())))
print(f"VRAM 사용 {(torch.cuda.mem_get_info()[1]-torch.cuda.mem_get_info()[0])/1e9:.1f} GB")

## 7. 학습 — 옵티마이저·스케줄러까지 저장하며 이어서 합니다

기존 노트북은 어댑터만 저장해서, 세션이 끊기면 AdamW 모멘텀과 학습률 스케줄이
초기화됐습니다. 여기서는 `train_state.pt` 에 전부 저장해 정확히 이어갑니다.

`SAVE_EVERY`(기본 20) optimizer step 마다 저장하므로, Colab이 갑자기 끊겨도
최대 20 step(약 3분)만 잃습니다.

In [ ]:
import time

# 정답 위치 1개에 대한 CE를 샘플 가중치로 가중평균 (위치 t의 logits가 t+1을 예측)
def masked_weighted_loss(logits, labels, last, weight):
    B = logits.size(0); r = torch.arange(B, device=logits.device)
    ce = F.cross_entropy(logits[r, last-1, :].float(), labels[r, last], reduction="none")
    return (ce * weight).sum() / weight.sum().clamp(min=1e-8)

params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=FT_LR, weight_decay=0.01)
sch = get_cosine_schedule_with_warmup(opt, max(1, int(TARGET_STEPS*0.03)), TARGET_STEPS)

# ★ fp16(T4)에서는 GradScaler가 필수입니다.
# fp16의 최소 정규값은 6e-5 라, lr 1e-4 규모의 작은 그래디언트가 그냥 0으로
# 언더플로합니다. 그러면 loss는 멀쩡한데 가중치가 안 움직이는 '조용한 무학습'이
# 됩니다. NaN 검사로는 절대 못 잡습니다. scaler가 loss를 키워서 이를 막습니다.
USE_SCALER = (COMPUTE_DTYPE == torch.float16)
scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
print(f"GradScaler: {'켜짐 (fp16)' if USE_SCALER else '꺼짐 (bf16은 불필요)'}")

# ---------- 상태 복원 ----------
if STATE_PATH.exists():
    st = torch.load(STATE_PATH, map_location="cpu")
    opt.load_state_dict(st["optimizer"])
    done = st["global_step"]; hist = st.get("hist", [])
    if st.get("target") == TARGET_STEPS:
        sch.load_state_dict(st["scheduler"])
    else:
        # TARGET_STEPS 를 바꿨으면 스케줄 모양이 달라지므로 새로 만들어 앞으로 감습니다
        print(f"⚠️ TARGET_STEPS 가 {st.get('target')} → {TARGET_STEPS} 로 바뀌었습니다. "
              "스케줄을 새로 만듭니다 (옵티마이저 상태는 유지)")
        for _ in range(done): sch.step()
    if "scaler" in st and USE_SCALER: scaler.load_state_dict(st["scaler"])
    print(f"▶ 이어서 시작: {done}/{TARGET_STEPS} step (이전 세션 {st.get('sessions',0)}회)")
else:
    done, hist = DONE_BEFORE, []
    for _ in range(done): sch.step()      # 데스크탑에서 한 만큼 스케줄만 앞으로 감기
    st = {"sessions": 0}
    print(f"▶ 새로 시작: 데스크탑 {done} step 이후부터 (옵티마이저는 새 상태)")
print(f"  현재 학습률 {sch.get_last_lr()[0]:.2e} / 목표 {TARGET_STEPS} step")

if done >= TARGET_STEPS:
    print("🎉 목표 step에 도달했습니다. 셀 8(재평가)로 가세요.")
else:
    loader = DataLoader(VQADataset(tr_part, train=True, aug=True), batch_size=FT_BATCH,
                        shuffle=True, collate_fn=Collator(processor, True), num_workers=2)

    def save_state(step, sessions):
        model.save_pretrained(ADAPTER_OUT)
        d = {"optimizer": opt.state_dict(), "scheduler": sch.state_dict(),
             "global_step": step, "hist": hist, "sessions": sessions,
             "target": TARGET_STEPS}
        if USE_SCALER: d["scaler"] = scaler.state_dict()
        torch.save(d, STATE_PATH)

    # ★ '조용한 무학습' 감지용 기준값 — 학습 전 LoRA 가중치의 절대합
    def lora_sum():
        return sum(float(p.detach().abs().sum()) for n, p in model.named_parameters()
                   if "lora_" in n and p.requires_grad)
    w0 = lora_sum()

    model.train(); opt.zero_grad(set_to_none=True)
    t0, run, seen, micro, nan_hits = time.time(), 0.0, 0, 0, 0
    start_step, stop, checked = done, False, False
    pbar = tqdm(total=TARGET_STEPS-done, desc="QLoRA", unit="step")
    while not stop:
        for batch in loader:
            w = batch.pop("_weight").to(device); last = batch.pop("_last").to(device)
            batch = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}
            lab = batch.pop("labels")
            loss = masked_weighted_loss(model(**batch).logits, lab, last, w)
            if not torch.isfinite(loss):
                nan_hits += 1; opt.zero_grad(set_to_none=True); micro = 0
                if nan_hits <= 3: tqdm.write(f"⚠️ loss가 {loss.item()} — 이 배치를 건너뜁니다")
                if nan_hits > 50:
                    tqdm.write("❌ NaN이 50회를 넘었습니다. fp16 불안정으로 보입니다 — "
                               "FT_LR 을 5e-5 로 낮춰 다시 시도하세요."); stop = True; break
                continue
            scaler.scale(loss / FT_ACCUM).backward(); run += loss.item(); seen += 1; micro += 1
            if micro == FT_ACCUM:
                scaler.unscale_(opt)                     # clip 전에 스케일 되돌리기
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(opt); scaler.update(); sch.step()
                opt.zero_grad(set_to_none=True)
                done += 1; micro = 0

                # 5 step 뒤 가중치가 실제로 움직였는지 확인 (fp16 언더플로 조기 발견)
                if not checked and done - start_step >= 5:
                    checked = True
                    moved = abs(lora_sum() - w0)
                    tqdm.write(f"가중치 이동량 {moved:.2f} "
                               + ("✅ 학습되고 있습니다" if moved > 1e-3 else
                                  "❌ 가중치가 안 움직입니다 — fp16 언더플로. "
                                  "런타임을 재시작하고 FT_LR을 3e-4로 올려 보세요"))
                    if moved <= 1e-3: stop = True; break
                hist.append(run/max(seen,1)); run, seen = 0.0, 0
                pbar.update(1)
                pbar.set_postfix({"loss": f"{np.mean(hist[-20:]):.3f}", "step": done,
                                  "lr": f"{sch.get_last_lr()[0]:.1e}"})
                if done % SAVE_EVERY == 0: save_state(done, st.get("sessions",0)+1)
                if done >= TARGET_STEPS:
                    tqdm.write(f"\n🎉 목표 {TARGET_STEPS} step 도달"); stop = True; break
                if (time.time()-t0)/60 > FT_MAX_MIN:
                    tqdm.write(f"\n⏰ {FT_MAX_MIN}분 경과 — 이번 세션 종료"); stop = True; break
    pbar.close()
    save_state(done, st.get("sessions",0)+1)
    el = (time.time()-t0)/60
    print(f"\n이번 세션 {done-start_step} step / {el:.0f}분 "
          f"({el*60/max(done-start_step,1):.0f} 초/step)")
    print(f"누적 {done}/{TARGET_STEPS} step  (loss 최근20 평균 {np.mean(hist[-20:]):.3f})")
    print(f"저장: {ADAPTER_OUT.name} + {STATE_PATH.name}")
    if done < TARGET_STEPS:
        rem = (TARGET_STEPS-done) * el*60/max(done-start_step,1) / 60
        print(f"남은 {TARGET_STEPS-done} step -> 약 {rem:.0f}분 = 세션 {math.ceil(rem/FT_MAX_MIN)}회 더")

## 8. 재평가 — 좋아졌을 때만 채택하세요 (약 8분)

홀드아웃 609건으로 다시 잽니다. 비교 기준은 **로컬에서 확인된 값**입니다.

| 구성 | 홀드아웃 609 | LB |
|---|---|---|
| zero-shot + 텍스트 | 0.8424 | 0.84430 |
| ft(104step) + zs + 텍스트 | *(로컬 tta_ft_check.py 참고)* | 0.86322 |

**609건 홀드아웃은 LB 오프셋이 ~0** 이므로 여기 숫자를 그대로 LB로 보셔도 됩니다.

In [ ]:
@torch.no_grad()
def predict_probs(df, bs=4, desc="eval"):
    ds = VQADataset(df, train=False, aug=False)
    dl = DataLoader(ds, batch_size=bs, shuffle=False,
                    collate_fn=Collator(processor, False), num_workers=2)
    model.eval(); out = []; cid = CHOICE_IDS.to(device)
    for batch in tqdm(dl, desc=desc):
        batch = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}
        lg = model(**batch).logits[:, -1, :]
        out.append(F.softmax(lg[:, cid].float(), -1).cpu())
    model.train()
    return torch.cat(out).numpy().astype(np.float64)

p_new = predict_probs(va_part, desc="홀드아웃 609")
np.save(WORK / f"vlm7b_probs_holdout609_ft{done}.npy", p_new)

gold = va_part["answer"].astype(str).str.strip().values
mcnt = is_counting(va_part["question"]).values
def acc(p, m=None):
    pr = CH[p.argmax(1)]
    return (pr == gold).mean() if m is None else (pr[m] == gold[m]).mean()

print("\n" + "="*56)
print(f"학습 {done} step 후 홀드아웃 609건")
print(f"  파인튜닝 단독  {acc(p_new):.4f}  (카운팅 {acc(p_new,mcnt):.4f} / 기타 {acc(p_new,~mcnt):.4f})")

# 텍스트/zero-shot 확률이 Drive에 있으면 최종 구성까지 계산
def norm(p): return p / p.sum(1, keepdims=True)
def bl(pv, pt, a=0.75): return norm(np.clip(pv,1e-9,1)**a * np.clip(pt,1e-9,1)**(1-a))
tx, zs = DRIVE/"text_probs_holdout.npy", DRIVE/"vlm_probs_holdout_full_7B_4bit_640px_zs_nodev.npy"
if tx.exists() and zs.exists():
    p_txt, p_zs = np.load(tx).astype(np.float64), np.load(zs).astype(np.float64)
    print(f"  + 텍스트       {acc(bl(p_new,p_txt)):.4f}")
    print(f"  + zs 평균+텍스트 {acc(bl(norm(p_new+p_zs),p_txt)):.4f}   ← 제출 구성 (LB 0.86322와 비교)")
else:
    print("  (text_probs_holdout.npy / vlm_probs_holdout_...npy 를 Drive에 올리면 최종 구성까지 나옵니다)")
print("="*56)
print("좋아졌으면 Drive의 colab_train/lora_colab_latest 를 내려받아")
print("로컬에서 `python infer_ft.py --test` 를 돌리세요.")